# 01 Preprocessing

Notebook này chứa pipeline chính: đọc video, chạy MediaPipe Face Mesh, tạo ROI/mask, chuẩn hóa ground truth, và ghi manifest. Module `airppg.preprocessing.*` chỉ giữ helper nhỏ có thể tái sử dụng.

## 1. Config

In [1]:
from __future__ import annotations

import json
import shutil
import tempfile
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path
import sys
from typing import Any


def find_project_root(start: Path | None = None) -> Path:
    """Find repo root from either repository root or notebooks/ working directory."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "airppg").exists():
            return candidate
    raise RuntimeError(f"Cannot find project root from {current}")


PROJECT_ROOT = find_project_root()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from IPython.display import Image, display
from tqdm import tqdm

from airppg.paths import portable_config_path, resolve_input_path, resolve_project_root, write_json
from airppg.preprocessing.config import PreprocessingBatchConfig, PreprocessingConfig
from airppg.preprocessing.dataset import (
    assign_dataset_splits,
    discover_dataset_videos,
    manifest_item_for_paths,
    sample_id_for_video,
    video_output_dir,
)
from airppg.preprocessing.ground_truth import save_normalized_ground_truth
from airppg.preprocessing.roi import (
    EXPECTED_LANDMARKS,
    ROI_LANDMARKS,
    clip_polygon,
    draw_rois,
    extract_roi_polygons,
    landmarks_to_pixels,
    pack_roi_polygons,
)
from airppg.schemas import DATASET_NAME, PREPROCESSING_MANIFEST_SCHEMA_VERSION, ROI_NAMES

USER_DATASET_ROOT = Path("datasets/UBFC_DATASET")
USER_OUTPUT_DIR = Path("outputs/preprocessing")
USER_VIDEO_PATH: Path | None = None

PROCESS_ALL_VIDEOS = True
BATCH_MAX_VIDEOS: int | None = None
SKIP_EXISTING = False
SPLIT_RATIOS = {"train": 0.8, "val": 0.1, "test": 0.1}
SPLIT_SEED = 42

TARGET_FPS: float | None = None
RESIZE_WIDTH: int | None = 640
MAX_FRAMES: int | None = None
SAVE_OVERLAY_VIDEO = False
OVERLAY_SAMPLE_COUNT = 8
MIN_ROI_AREA_PX = 100.0

config = PreprocessingConfig(
    target_fps=TARGET_FPS,
    resize_width=RESIZE_WIDTH,
    max_frames=MAX_FRAMES,
    save_overlay_video=SAVE_OVERLAY_VIDEO,
    overlay_sample_count=OVERLAY_SAMPLE_COUNT,
    min_roi_area_px=MIN_ROI_AREA_PX,
)
batch_config = PreprocessingBatchConfig(
    dataset_root=USER_DATASET_ROOT,
    output_root=USER_OUTPUT_DIR,
    process_all_videos=PROCESS_ALL_VIDEOS,
    batch_max_videos=BATCH_MAX_VIDEOS,
    skip_existing=SKIP_EXISTING,
    split_ratios=SPLIT_RATIOS,
    split_seed=SPLIT_SEED,
)

display(pd.DataFrame([config.to_dict()]).T.rename(columns={0: "value"}))
display(pd.DataFrame([batch_config.to_dict()]).T.rename(columns={0: "value"}))

,value
target_fps,None
resize_width,640
max_frames,None
save_overlay_video,False
overlay_sample_count,8
min_roi_area_px,100.0
schema_version,preprocessing_roi_v1


,value
process_all_videos,True
batch_max_videos,None
skip_existing,False


## 2. Local helpers

In [2]:
@dataclass(slots=True)
class VideoMetadata:
    path: str
    fps: float
    frame_count: int
    width: int
    height: int
    duration_sec: float


def path_has_non_ascii(path: Path) -> bool:
    try:
        str(path).encode("ascii")
    except UnicodeEncodeError:
        return True
    return False


def configure_mediapipe_resource_root() -> Path | None:
    package_root = Path(mp.__file__).resolve().parent
    graph_path = package_root / "modules" / "face_landmark" / "face_landmark_front_cpu.binarypb"
    if not graph_path.exists() or not path_has_non_ascii(package_root):
        return None
    import mediapipe.python.solution_base as solution_base

    target_root = Path(tempfile.gettempdir()) / "airppg_mediapipe_resources" / "mediapipe"
    target_modules = target_root / "modules"
    target_graph = target_modules / "face_landmark" / "face_landmark_front_cpu.binarypb"
    if not target_graph.exists():
        target_root.mkdir(parents=True, exist_ok=True)
        shutil.copytree(package_root / "modules", target_modules, dirs_exist_ok=True)
    (target_root / "python").mkdir(parents=True, exist_ok=True)
    solution_base.__file__ = str(target_root / "python" / "solution_base.py")
    return target_root


def read_video_metadata(video_path: Path) -> VideoMetadata:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")
    try:
        fps = float(cap.get(cv2.CAP_PROP_FPS)) or 0.0
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    finally:
        cap.release()
    duration_sec = frame_count / fps if fps > 0 else 0.0
    return VideoMetadata(str(video_path), fps, frame_count, width, height, duration_sec)


def resize_frame(frame: np.ndarray, resize_width: int | None) -> np.ndarray:
    if resize_width is None or frame.shape[1] == resize_width:
        return frame
    scale = resize_width / float(frame.shape[1])
    return cv2.resize(frame, (resize_width, int(round(frame.shape[0] * scale))), interpolation=cv2.INTER_AREA)


def should_process_frame(frame_idx: int, source_fps: float, target_fps: float | None) -> bool:
    if target_fps is None or target_fps <= 0 or source_fps <= 0 or target_fps >= source_fps:
        return True
    step = source_fps / target_fps
    return abs((frame_idx / step) - round(frame_idx / step)) < (0.5 / step)


def write_image(path: Path, image_rgb: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(path), cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR))

## 3. Main preprocessing pipeline

In [3]:
PROJECT_ROOT = resolve_project_root(PROJECT_ROOT)
DATASET_ROOT = resolve_input_path(batch_config.dataset_root, PROJECT_ROOT)
OUTPUT_ROOT = resolve_input_path(batch_config.output_root, PROJECT_ROOT)
DATASET_ROOT_HINT = portable_config_path(batch_config.dataset_root, PROJECT_ROOT)
OUTPUT_ROOT_HINT = portable_config_path(batch_config.output_root, PROJECT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

all_videos = discover_dataset_videos(DATASET_ROOT)
selected_videos = all_videos[: batch_config.batch_max_videos] if batch_config.batch_max_videos is not None else all_videos
split_assignments = assign_dataset_splits(all_videos, DATASET_ROOT, batch_config.ratios(), batch_config.split_seed)
if USER_VIDEO_PATH is not None:
    selected_videos = [resolve_input_path(USER_VIDEO_PATH, PROJECT_ROOT)]

manifest_items: list[dict[str, Any]] = []
mediapipe_root = configure_mediapipe_resource_root()
if mediapipe_root is not None:
    print(f"MEDIAPIPE_RESOURCE_ROOT = {mediapipe_root}")

for video_path in tqdm(selected_videos, desc="Preprocessing samples", unit="sample"):
    sample_id = sample_id_for_video(video_path, DATASET_ROOT)
    split = split_assignments.get(video_path, "train")
    output_dir = video_output_dir(OUTPUT_ROOT, split, sample_id)
    output_dir.mkdir(parents=True, exist_ok=True)
    try:
        video_metadata = read_video_metadata(video_path)
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            raise FileNotFoundError(f"Cannot open video: {video_path}")

        frame_indices: list[int] = []
        timestamps: list[float] = []
        landmarks_all: list[np.ndarray] = []
        roi_polygons_all: list[np.ndarray] = []
        roi_masks_all: list[np.ndarray] = []
        valid_all: list[bool] = []
        errors: list[str] = []
        overlay_frames: list[tuple[int, np.ndarray]] = []

        face_mesh = mp.solutions.face_mesh.FaceMesh(
            static_image_mode=False,
            max_num_faces=1,
            refine_landmarks=True,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5,
        )
        try:
            frame_iter = range(video_metadata.frame_count if video_metadata.frame_count > 0 else 10**12)
            for frame_idx in frame_iter:
                ok, frame_bgr = cap.read()
                if not ok:
                    break
                if config.max_frames is not None and len(frame_indices) >= config.max_frames:
                    break
                if not should_process_frame(frame_idx, video_metadata.fps, config.target_fps):
                    continue
                frame_rgb = cv2.cvtColor(resize_frame(frame_bgr, config.resize_width), cv2.COLOR_BGR2RGB)
                height, width = frame_rgb.shape[:2]
                frame_indices.append(frame_idx)
                timestamps.append(frame_idx / video_metadata.fps if video_metadata.fps > 0 else float(frame_idx))

                landmarks = np.full((EXPECTED_LANDMARKS, 2), np.nan, dtype=np.float32)
                roi_masks = np.zeros((len(ROI_NAMES), height, width), dtype=bool)
                valid = False
                error = "no_face_detected"
                polygons = {name: np.empty((0, 2), dtype=np.float32) for name in ROI_NAMES}
                try:
                    result = face_mesh.process(frame_rgb)
                    if result.multi_face_landmarks:
                        landmarks_px = landmarks_to_pixels(result.multi_face_landmarks[0], width, height)
                        landmarks[: min(EXPECTED_LANDMARKS, landmarks_px.shape[0])] = landmarks_px[:EXPECTED_LANDMARKS]
                        polygons, roi_masks, valid, roi_error = extract_roi_polygons(
                            landmarks_px,
                            width,
                            height,
                            config.min_roi_area_px,
                        )
                        error = "" if valid else (roi_error or "invalid_roi")
                except Exception as exc:
                    error = f"exception:{type(exc).__name__}:{exc}"

                roi_polygons = pack_roi_polygons(polygons)
                landmarks_all.append(landmarks)
                roi_polygons_all.append(roi_polygons)
                roi_masks_all.append(roi_masks)
                valid_all.append(valid)
                errors.append(error)
                if config.overlay_sample_count > 0 and len(overlay_frames) < config.overlay_sample_count:
                    overlay_frames.append((frame_idx, draw_rois(frame_rgb, polygons, valid)))
        finally:
            face_mesh.close()
            cap.release()

        if roi_masks_all:
            roi_masks_array = np.stack(roi_masks_all).astype(bool)
            max_points = max(p.shape[1] for p in roi_polygons_all)
            roi_polygons_array = np.full((len(roi_polygons_all), len(ROI_NAMES), max_points, 2), np.nan, dtype=np.float32)
            for idx, polygons in enumerate(roi_polygons_all):
                roi_polygons_array[idx, :, : polygons.shape[1], :] = polygons
        else:
            roi_masks_array = np.zeros((0, len(ROI_NAMES), 0, 0), dtype=bool)
            roi_polygons_array = np.zeros((0, len(ROI_NAMES), 0, 2), dtype=np.float32)

        np.savez_compressed(
            output_dir / "roi_data.npz",
            frame_indices=np.asarray(frame_indices, dtype=np.int32),
            timestamps=np.asarray(timestamps, dtype=np.float32),
            landmarks=np.asarray(landmarks_all, dtype=np.float32),
            landmarks_px=np.asarray(landmarks_all, dtype=np.float32),
            roi_polygons=roi_polygons_array.astype(np.float32),
            roi_polygons_px=roi_polygons_array.astype(np.float32),
            roi_masks=roi_masks_array.astype(np.uint8),
            valid=np.asarray(valid_all, dtype=bool),
            error_reasons=np.asarray(errors, dtype=str),
            roi_names=np.asarray(ROI_NAMES, dtype=str),
        )

        overlay_dir = output_dir / "overlay_samples"
        for sample_idx, (frame_idx, overlay) in enumerate(overlay_frames):
            write_image(overlay_dir / f"frame_{sample_idx:03d}_{frame_idx:06d}.png", overlay)

        ground_truth = save_normalized_ground_truth(video_path, DATASET_ROOT, DATASET_ROOT_HINT, output_dir)
        metadata_payload = {
            "schema_version": config.schema_version,
            "dataset_name": DATASET_NAME,
            "video": asdict(video_metadata),
            "config": config.to_dict(),
            "roi_names": list(ROI_NAMES),
            "processed_frame_count": len(frame_indices),
            "valid_frame_count": int(np.sum(valid_all)),
            "invalid_frame_count": int(len(valid_all) - np.sum(valid_all)),
            "failure_counts": dict(Counter(reason for reason in errors if reason)),
            "ground_truth": ground_truth,
            "artifacts": {
                "roi_npz": "roi_data.npz",
                "metadata_json": "metadata.json",
                "ground_truth_npz": "ground_truth.npz" if ground_truth.get("available") else None,
                "overlay_samples_dir": "overlay_samples",
            },
        }
        write_json(output_dir / "metadata.json", metadata_payload)
        manifest_items.append(manifest_item_for_paths(video_path, DATASET_ROOT, DATASET_ROOT_HINT, OUTPUT_ROOT, split, sample_id, "processed"))
    except Exception as exc:
        manifest_items.append(manifest_item_for_paths(video_path, DATASET_ROOT, DATASET_ROOT_HINT, OUTPUT_ROOT, split, sample_id, "failed", str(exc)))

processed = sum(item["status"] == "processed" for item in manifest_items)
failed = sum(item["status"] == "failed" for item in manifest_items)
manifest = {
    "schema_version": PREPROCESSING_MANIFEST_SCHEMA_VERSION,
    "dataset_root_hint": DATASET_ROOT_HINT,
    "dataset_readme_relpath": f"{DATASET_ROOT_HINT}/readme.txt" if DATASET_ROOT_HINT else "readme.txt",
    "output_root_hint": OUTPUT_ROOT_HINT,
    "splits": ["train", "val", "test"],
    "split_policy": "stable_hash_sample_level_stratified_by_source_dataset_split_when_possible",
    "split_ratios": batch_config.ratios(),
    "split_seed": batch_config.split_seed,
    "config": config.to_dict(),
    "batch_config": batch_config.to_dict(),
    "counts": {
        "total": len(manifest_items),
        "processed": processed,
        "skipped": 0,
        "failed": failed,
        "total_discovered_videos": len(all_videos),
        "selected_video_count": len(selected_videos),
    },
    "items": manifest_items,
}
write_json(OUTPUT_ROOT / "preprocessing_dataset_manifest.json", manifest)

display(pd.DataFrame([manifest["counts"]]))
display(pd.DataFrame(manifest_items))
if failed:
    raise RuntimeError(f"Preprocessing failed for {failed} sample(s).")

MEDIAPIPE_RESOURCE_ROOT = C:\Users\ducvu\AppData\Local\Temp\airppg_mediapipe_resources\mediapipe


Preprocessing samples:   0%|          | 0/49 [00:00<?, ?sample/s]d:\Workspace\FPTspace\Kỳ 5\DSR\lightweight_rppg\.venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
Preprocessing samples: 100%|██████████| 49/49 [24:01<00:00, 29.42s/sample]


,total,processed,skipped,failed,total_discovered_videos,selected_video_count
0,49,49,0,0,49,49


,sample_id,split,status,source_dataset,subject,video_relpath,output_dir,roi_npz,metadata_json,ground_truth_npz
0,ubfc_d1_10-gt_vid_3440368f,train,processed,DATASET_1,10-gt,datasets/UBFC_DATASET/DATASET_1/10-gt/vid.avi,train/ubfc_d1_10-gt_vid_3440368f,train/ubfc_d1_10-gt_vid_3440368f/roi_data.npz,train/ubfc_d1_10-gt_vid_3440368f/metadata.json,train/ubfc_d1_10-gt_vid_3440368f/ground_truth.npz
1,ubfc_d1_11-gt_vid_97458e9b,train,processed,DATASET_1,11-gt,datasets/UBFC_DATASET/DATASET_1/11-gt/vid.avi,train/ubfc_d1_11-gt_vid_97458e9b,train/ubfc_d1_11-gt_vid_97458e9b/roi_data.npz,train/ubfc_d1_11-gt_vid_97458e9b/metadata.json,train/ubfc_d1_11-gt_vid_97458e9b/ground_truth.npz
2,ubfc_d1_12-gt_vid-040_eddd5a70,test,processed,DATASET_1,12-gt,datasets/UBFC_DATASET/DATASET_1/12-gt/vid-040.avi,test/ubfc_d1_12-gt_vid-040_eddd5a70,test/ubfc_d1_12-gt_vid-040_eddd5a70/roi_data.npz,test/ubfc_d1_12-gt_vid-040_eddd5a70/metadata.json,test/ubfc_d1_12-gt_vid-040_eddd5a70/ground_tru...
3,ubfc_d1_5-gt_vid-049_a22d895e,train,processed,DATASET_1,5-gt,datasets/UBFC_DATASET/DATASET_1/5-gt/vid-049.avi,train/ubfc_d1_5-gt_vid-049_a22d895e,train/ubfc_d1_5-gt_vid-049_a22d895e/roi_data.npz,train/ubfc_d1_5-gt_vid-049_a22d895e/metadata.json,train/ubfc_d1_5-gt_vid-049_a22d895e/ground_tru...
4,ubfc_d1_6-gt_vid-045_d5d05981,train,processed,DATASET_1,6-gt,datasets/UBFC_DATASET/DATASET_1/6-gt/vid-045.avi,train/ubfc_d1_6-gt_vid-045_d5d05981,train/ubfc_d1_6-gt_vid-045_d5d05981/roi_data.npz,train/ubfc_d1_6-gt_vid-045_d5d05981/metadata.json,train/ubfc_d1_6-gt_vid-045_d5d05981/ground_tru...
5,ubfc_d1_7-gt_vid-042_ec139516,train,processed,DATASET_1,7-gt,datasets/UBFC_DATASET/DATASET_1/7-gt/vid-042.avi,train/ubfc_d1_7-gt_vid-042_ec139516,train/ubfc_d1_7-gt_vid-042_ec139516/roi_data.npz,train/ubfc_d1_7-gt_vid-042_ec139516/metadata.json,train/ubfc_d1_7-gt_vid-042_ec139516/ground_tru...
6,ubfc_d1_8-gt_vid_c4e9d008,val,processed,DATASET_1,8-gt,datasets/UBFC_DATASET/DATASET_1/8-gt/vid.avi,val/ubfc_d1_8-gt_vid_c4e9d008,val/ubfc_d1_8-gt_vid_c4e9d008/roi_data.npz,val/ubfc_d1_8-gt_vid_c4e9d008/metadata.json,val/ubfc_d1_8-gt_vid_c4e9d008/ground_truth.npz
7,ubfc_d2_subject1_vid_f522da8f,train,processed,DATASET_2,subject1,datasets/UBFC_DATASET/DATASET_2/subject1/vid.avi,train/ubfc_d2_subject1_vid_f522da8f,train/ubfc_d2_subject1_vid_f522da8f/roi_data.npz,train/ubfc_d2_subject1_vid_f522da8f/metadata.json,train/ubfc_d2_subject1_vid_f522da8f/ground_tru...
8,ubfc_d2_subject10_vid_142da79b,train,processed,DATASET_2,subject10,datasets/UBFC_DATASET/DATASET_2/subject10/vid.avi,train/ubfc_d2_subject10_vid_142da79b,train/ubfc_d2_subject10_vid_142da79b/roi_data.npz,train/ubfc_d2_subject10_vid_142da79b/metadata....,train/ubfc_d2_subject10_vid_142da79b/ground_tr...
9,ubfc_d2_subject11_vid_86a002c4,val,processed,DATASET_2,subject11,datasets/UBFC_DATASET/DATASET_2/subject11/vid.avi,val/ubfc_d2_subject11_vid_86a002c4,val/ubfc_d2_subject11_vid_86a002c4/roi_data.npz,val/ubfc_d2_subject11_vid_86a002c4/metadata.json,val/ubfc_d2_subject11_vid_86a002c4/ground_trut...


## 4. Overlay sample

In [4]:
first_processed = next((item for item in manifest.get("items", []) if item.get("status") == "processed"), None)
if first_processed is not None:
    sample_dir = OUTPUT_ROOT / first_processed["output_dir"]
    overlays = sorted((sample_dir / "overlay_samples").glob("*.png"))
    if overlays:
        display(Image(filename=str(overlays[0])))